# 🎯 Lab 4: Stochastic Monte Carlo & Landing Dispersion Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blaze505050/drone-digital-twin/blob/main/examples/labs/Lab4_Stochastic_Monte_Carlo_and_Dispersion.ipynb)

Welcome to **Lab 4 of the DronePy Aerospace & Robotics Curriculum**!
In this lab, you will master **Uncertainty Quantification (UQ)**, probabilistic parameter perturbation, batch trajectory simulations, and the computation of **Circular Error Probable (CEP50 / CEP95)** landing footprints for autonomous precision operations.

---
### 🎯 Learning Objectives
1. Define statistical distributions (Gaussian, Uniform) for vehicle mass and aerodynamic drag.
2. Execute multi-run Monte Carlo simulations using DronePy's `MonteCarlo` engine.
3. Compute bivariate landing error statistics, standard deviations, and CEP50 / CEP95 circles.
4. Assess whether an autonomous drone meets safety criteria for landing on constrained vertipads.


In [ ]:
# Setup dependencies
try:
    import dronepy
except ImportError:
    !pip install -q git+https://github.com/blaze505050/drone-digital-twin.git
    import dronepy

import numpy as np
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f"DronePy Version: {dronepy.__version__}")


---
## 1. Mathematical Theory: Dispersion & Circular Error Probable (CEP)

In physical flight tests, vehicle properties and environmental parameters are never known with absolute certainty.
We model parameters as random variables:
- Mass uncertainty: $m \sim \mathcal{N}(\mu_m, \sigma_m^2)$
- Parasitic drag coefficient: $C_d \sim \mathcal{U}(C_{d,\text{min}}, C_{d,\text{max}})$

### Landing Error & Dispersion Radii
For $N$ simulation runs, let $(x_i, y_i)$ be the landing position of run $i$ relative to the intended target:
$$R_i = \sqrt{(x_i - x_{\text{target}})^2 + (y_i - y_{\text{target}})^2}$$

### Circular Error Probable (CEP)
- **CEP50**: The radius of a circle centered at the target containing **50%** of all landing attempts (the median radial error).
- **CEP95**: The radius of a circle containing **95%** of all landing attempts.
For bivariate normal dispersion with radial symmetry:
$$\text{CEP}_{50} \approx 0.5887 (\sigma_x + \sigma_y)$$
$$\text{CEP}_{95} \approx 1.2238 (\sigma_x + \sigma_y)$$


In [ ]:
# 2. Running a Monte Carlo Simulation with DronePy
# Let's perform a 6-run Monte Carlo study perturbing drone mass
drone = dronepy.Drone.quadcopter(mass=1.5)
mc = dronepy.MonteCarlo(drone=drone, num_simulations=6, seed=42)

# Add uncertain parameter
mc.add_parameter("mass", dronepy.Distribution.normal(mean=1.50, std_dev=0.08))

# Execute batch simulations
mc_result = mc.run(duration=1.5, parallel=False)
summary = mc_result.summary()

print(f"Completed {mc_result.runs} Monte Carlo iterations.")
print(f"CEP50 Radius: {summary.get('cep50_m', 0.0):.3f} m")
print(f"CEP95 Radius: {summary.get('cep95_m', 0.0):.3f} m")


---
## 📝 Student Exercise: Autonomous Vertipad Landing Clearance

### Mission Briefing:
An autonomous drone delivery operator wants to certify operations on an urban rooftop vertipad with a usable landing radius of **$R_{\text{pad}} = 2.50\text{ m}$**.
Aviation safety regulations stipulate that the vehicle's **CEP95 footprint must not exceed the pad radius**:
$$\text{CEP}_{95} \le 2.50\text{ m}$$

### Your Tasks:
1. Create a baseline $1.60\text{ kg}$ quadcopter.
2. Configure a `MonteCarlo` campaign with 6 simulations.
3. Perturb the vehicle mass using $\mathcal{N}(\mu=1.60, \sigma=0.10)\text{ kg}$.
4. Compute the empirical 95th percentile landing radius.
5. Determine whether the vehicle meets the vertipad landing certification!


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STUDENT SOLUTION CELL - Execute vertipad certification analysis:
# ══════════════════════════════════════════════════════════════════
cert_drone = dronepy.Drone.quadcopter(mass=1.60)
mc_cert = dronepy.MonteCarlo(drone=cert_drone, num_simulations=6, seed=101)
mc_cert.add_parameter("mass", dronepy.Distribution.normal(mean=1.60, std_dev=0.10))

res_cert = mc_cert.run(duration=1.5, parallel=False)
pad_radius_limit = 2.50
cep95 = res_cert.landing_dispersion_radius_95

passes_certification = cep95 <= pad_radius_limit

print(f"Evaluated Runs:          {res_cert.runs}")
print(f"Computed CEP95 Radius:   {cep95:.3f} m")
print(f"Vertipad Limit Radius:   {pad_radius_limit:.2f} m")
print(f"Certification Status:    {'PASSED' if passes_certification else 'FAILED'}")

# Verification Assertions
assert res_cert.runs == 6, "Must execute all 6 runs"
assert cep95 >= 0.0, "CEP95 must be non-negative"
print("SUCCESS: Lab 4 Monte Carlo dispersion verified successfully!")
